# Academic Reference Staleness Check

This notebook checks whether cached academic references are stale in `rag_data/academic_references.db` and provides actionable re-validation commands.

A reference is treated as stale if **any** of these apply:
- `expires_at` is in the past
- Cache age is above the staleness threshold (default 30 days)
- `link_status` starts with `stale_`
- `status` is `failed`

In [ ]:
from __future__ import annotations

from datetime import timezone
from pathlib import Path
import sqlite3

import pandas as pd

STALE_THRESHOLD_DAYS = 30

def resolve_db_path() -> Path:
    """Resolve the academic references DB path from common working directories."""
    candidates = [
        Path.cwd() / "rag_data" / "academic_references.db",
        Path.cwd().parent / "rag_data" / "academic_references.db",
        Path.cwd().parent.parent / "rag_data" / "academic_references.db",
    ]
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]

DB_PATH = resolve_db_path()
NOW_UTC = pd.Timestamp.now(tz=timezone.utc)

print(f"Database path: {DB_PATH}")
print(f"Staleness threshold (days): {STALE_THRESHOLD_DAYS}")

In [ ]:
DB_AVAILABLE = DB_PATH.exists()

if DB_AVAILABLE:
    print(f"✓ Found database: {DB_PATH}")
else:
    print("✗ academic_references.db not found.")
    print("Run academic ingestion first, for example:")
    print("  python scripts/ingest/ingest_academic.py --papers-dir data_raw/academic_papers")

In [ ]:
if DB_AVAILABLE:
    query = """
    SELECT
        ref_id,
        title,
        doi,
        status,
        reference_type,
        metadata_provider,
        link_status,
        cached_at,
        resolved_at,
        expires_at
    FROM cached_references
    """

    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query(query, conn)

    for column in ["cached_at", "resolved_at", "expires_at"]:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column], utc=True, errors="coerce")

    df["age_days"] = (NOW_UTC - df["cached_at"]).dt.total_seconds() / 86400
    df["age_days"] = df["age_days"].fillna(-1)

    df["is_expired"] = df["expires_at"].notna() & (df["expires_at"] < NOW_UTC)
    df["stale_by_age"] = df["age_days"] >= STALE_THRESHOLD_DAYS
    df["stale_link"] = df["link_status"].fillna("").str.startswith("stale_")
    df["failed_status"] = df["status"].fillna("").str.lower().eq("failed")

    df["is_stale"] = (
        df["is_expired"]
        | df["stale_by_age"]
        | df["stale_link"]
        | df["failed_status"]
    )

    summary = {
        "total_references": int(len(df)),
        "expired": int(df["is_expired"].sum()),
        "stale_by_age": int(df["stale_by_age"].sum()),
        "stale_link": int(df["stale_link"].sum()),
        "failed_status": int(df["failed_status"].sum()),
        "overall_stale": int(df["is_stale"].sum()),
    }

    print("Staleness summary")
    print("-" * 60)
    for key, value in summary.items():
        print(f"{key:>18}: {value}")
else:
    df = pd.DataFrame()
    print("Skipped: database is not available.")

In [ ]:
if not df.empty:
    stale_df = df[df["is_stale"]].copy()

    stale_df["risk_rank"] = (
        stale_df["is_expired"].astype(int) * 1000
        + stale_df["stale_link"].astype(int) * 100
        + stale_df["failed_status"].astype(int) * 10
    )

    stale_df = stale_df.sort_values(
        by=["risk_rank", "age_days"],
        ascending=[False, False],
    )

    display_columns = [
        "ref_id",
        "title",
        "doi",
        "status",
        "reference_type",
        "metadata_provider",
        "link_status",
        "age_days",
        "is_expired",
        "stale_by_age",
        "stale_link",
        "failed_status",
    ]

    print(f"Showing {min(len(stale_df), 50)} stale references (up to 50 rows)")
    display(stale_df[display_columns].head(50))
else:
    stale_df = pd.DataFrame()
    print("No reference data available to display.")

In [ ]:
print("Recommended re-validation commands")
print("-" * 60)

print("1) Revalidate stale references")
print("   python scripts/ingest/ingest_academic.py --revalidate stale --staleness-threshold 30")

print("\n2) Revalidate failed references")
print("   python scripts/ingest/ingest_academic.py --revalidate failed")

print("\n3) Revalidate online references")
print("   python scripts/ingest/ingest_academic.py --revalidate online")

print("\n4) Revalidate all references")
print("   python scripts/ingest/ingest_academic.py --revalidate all")

if not stale_df.empty:
    sample_ids = stale_df["ref_id"].dropna().astype(str).head(10).tolist()
    ids_str = " ".join(sample_ids)

    print("\n5) Revalidate specific reference IDs (sample)")
    print(f"   python scripts/ingest/ingest_academic.py --revalidate ids --ref-ids {ids_str}")

print("\nNext steps")
print("- Run the command that matches your stale pattern.")
print("- Re-run this notebook to confirm stale counts have dropped.")
print("- If stale links remain high, prioritise '--revalidate online' then '--revalidate stale'.")